Install dependencies

In [1]:
!pip install torch torchvision torchaudio -q
!pip install torch-geometric -q
!pip install gensim -q
!pip install scikit-learn -q
!pip install seaborn -q
!pip install networkx -q
!pip install pandas -q

Imports + GPU Setup

In [2]:
import os
import random
import time
import tracemalloc

import numpy as np
import pandas as pd
import networkx as nx

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from gensim.models import Word2Vec

from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

from torch_geometric.datasets import Planetoid
from torch_geometric.datasets import CitationFull
import torch_geometric.utils as pyg_utils

sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.5)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

Using device: cuda


Utility Functions

In [3]:
def measure_time_memory(func, *args, **kwargs):

    tracemalloc.start()

    start = time.time()

    result = func(*args, **kwargs)

    current, peak = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    return result, time.time() - start, peak / 1e6

Random Walks

In [4]:
def generate_random_walks(G, num_walks=5, walk_length=20):

    walks = []
    nodes = list(G.nodes())

    for _ in range(num_walks):
        random.shuffle(nodes)

        for node in nodes:
            walk = [str(node)]
            current = node

            for _ in range(walk_length - 1):
                neighbors = list(G.neighbors(current))
                if not neighbors:
                    break
                current = random.choice(neighbors)
                walk.append(str(current))

            walks.append(walk)

    return walks

DeepWalk

In [5]:
def train_deepwalk(G, emb_dim=128, epochs=5):

    walks = generate_random_walks(G)

    model = Word2Vec(
        walks,
        vector_size=emb_dim,
        window=5,
        min_count=0,
        sg=1,
        workers=4,
        epochs=epochs
    )

    num_nodes = G.number_of_nodes()

    embeddings = np.array([
        model.wv[str(i)] for i in range(num_nodes)
    ])

    return embeddings

CP-LSH Hashing

In [6]:
def compute_lsh(features, m=6, k=10):

    num_nodes, dim = features.shape

    random_vectors = np.random.randn(dim, m * k)

    random_vectors /= np.linalg.norm(random_vectors, axis=0)

    hashes = np.zeros((num_nodes, m), dtype=np.int64)

    for i in range(num_nodes):

        for h in range(m):

            bucket = 0

            for bit in range(k):

                idx = h * k + bit

                dot = np.dot(features[i], random_vectors[:, idx])

                if dot > 0:
                    bucket |= (1 << bit)

            bucket_id = h * (2 ** k) + bucket

            hashes[i, h] = bucket_id

    return hashes

CP-LSH Model

In [7]:
class CPLSH(nn.Module):

    def __init__(self, total_buckets, emb_dim):

        super().__init__()

        self.src_emb = nn.Embedding(total_buckets, emb_dim)

        self.tgt_emb = nn.Embedding(total_buckets, emb_dim)

        nn.init.xavier_uniform_(self.src_emb.weight)

        nn.init.xavier_uniform_(self.tgt_emb.weight)

    def forward(self, hashes):

        src = self.src_emb(hashes).mean(dim=1)

        tgt = self.tgt_emb(hashes).mean(dim=1)

        return src, tgt

SkipGram Pairs

In [8]:
def generate_skipgram_pairs(walks, window_size=5):

    pairs = []

    for walk in walks:

        walk = [int(x) for x in walk]

        for i, u in enumerate(walk):

            left = max(0, i - window_size)

            right = min(len(walk), i + window_size + 1)

            for j in range(left, right):

                if i == j:
                    continue

                v = walk[j]

                pairs.append((u, v))

    return pairs

Train CP-LSH

In [9]:
def train_cplsh(
    G,
    features,
    emb_dim=128,
    m=6,
    k=10,
    epochs=3,
    neg_samples=5,
    batch_size=512
):

    num_nodes = G.number_of_nodes()

    # ---------- LSH ----------
    hashes = compute_lsh(features, m, k)
    hashes = torch.tensor(hashes).long().to(DEVICE)

    total_buckets = m * (2 ** k)

    model = CPLSH(total_buckets, emb_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    # ---------- Generate SkipGram pairs ----------
    walks = generate_random_walks(G)
    pairs = generate_skipgram_pairs(walks)

    print("Total training pairs:", len(pairs))

    # Convert to tensor once
    pairs = torch.tensor(pairs).long()

    # ---------- Training ----------
    for epoch in range(epochs):

        permutation = torch.randperm(len(pairs))
        pairs = pairs[permutation]

        total_loss = 0

        for i in range(0, len(pairs), batch_size):

            batch = pairs[i:i+batch_size]
            u = batch[:,0].to(DEVICE)
            v = batch[:,1].to(DEVICE)

            optimizer.zero_grad()

            # --- Only compute embeddings for needed nodes ---
            src_u = model.src_emb(hashes[u]).mean(dim=1)
            tgt_v = model.tgt_emb(hashes[v]).mean(dim=1)

            pos_score = torch.sum(src_u * tgt_v, dim=1)
            pos_loss = -F.logsigmoid(pos_score).mean()

            # ---- Negative Sampling ----
            neg_nodes = torch.randint(
                0,
                num_nodes,
                (len(u), neg_samples),
                device=DEVICE
            )

            neg_emb = model.tgt_emb(hashes[neg_nodes]).mean(dim=2)

            src_expand = src_u.unsqueeze(1)

            neg_score = torch.sum(src_expand * neg_emb, dim=2)

            neg_loss = -F.logsigmoid(-neg_score).mean()

            loss = pos_loss + neg_loss

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

    model.eval()

    with torch.no_grad():
        final_emb = model.src_emb(hashes).mean(dim=1)

    return final_emb.cpu().numpy()

Node Classification

In [10]:
def node_classification(embeddings, labels):

    ratios = [0.1,0.2,0.3,0.4,0.5]

    results = []

    for ratio in ratios:

        scores = []

        for _ in range(5):

            X_train, X_test, y_train, y_test = train_test_split(
                embeddings,
                labels,
                train_size=ratio,
                stratify=labels
            )

            clf = LinearSVC(max_iter=5000)

            clf.fit(X_train, y_train)

            preds = clf.predict(X_test)

            f1 = f1_score(y_test, preds, average="micro")

            scores.append(f1)

        results.append((ratio, np.mean(scores)))

    return results

Tail vs Head Analysis

In [11]:
def tail_head_analysis(G, embeddings, labels):

    degrees = dict(G.degree())

    threshold = np.percentile(list(degrees.values()), 20)

    tail_nodes = [i for i,d in degrees.items() if d <= threshold]

    head_nodes = [i for i,d in degrees.items() if d > threshold]

    idx = np.arange(len(labels))

    train_idx, test_idx = train_test_split(
        idx,
        test_size=0.5,
        stratify=labels
    )

    clf = LinearSVC(max_iter=5000)

    clf.fit(embeddings[train_idx], labels[train_idx])

    preds = clf.predict(embeddings[test_idx])

    true = labels[test_idx]

    tail_mask = [i for i,x in enumerate(test_idx) if x in tail_nodes]
    head_mask = [i for i,x in enumerate(test_idx) if x in head_nodes]

    tail_f1 = f1_score(
        true[tail_mask],
        preds[tail_mask],
        average="micro"
    )

    head_f1 = f1_score(
        true[head_mask],
        preds[head_mask],
        average="micro"
    )

    return tail_f1, head_f1

Link Prediction

In [12]:
def link_prediction(G, embeddings):

    edges = list(G.edges())

    nodes = list(G.nodes())

    test_edges = random.sample(edges, int(0.2 * len(edges)))

    neg_edges = []

    while len(neg_edges) < len(test_edges):

        u, v = random.sample(nodes, 2)

        if not G.has_edge(u, v):

            neg_edges.append((u, v))

    y_true = [1] * len(test_edges) + [0] * len(neg_edges)

    y_scores = []

    for u, v in test_edges:

        y_scores.append(np.dot(embeddings[u], embeddings[v]))

    for u, v in neg_edges:

        y_scores.append(np.dot(embeddings[u], embeddings[v]))

    auc = roc_auc_score(y_true, y_scores)

    return auc

Degree Distribution Plot

In [13]:
def plot_degree_distribution(G, dataset_name):

    degrees = [d for n,d in G.degree()]

    plt.figure(figsize=(6,4))

    plt.hist(degrees, bins=50)

    plt.yscale("log")

    plt.xlabel("Degree")

    plt.ylabel("Frequency")

    plt.title(f"{dataset_name} Degree Distribution")

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_degree_distribution.png")

    plt.close()

t-SNE Visualization

In [14]:
def tsne_plot(embeddings, labels, dataset_name, method_name):

    tsne = TSNE(
      n_components=2,
      perplexity=30,
      n_iter=500,
      random_state=42
    )

    emb_2d = tsne.fit_transform(embeddings)

    plt.figure(figsize=(7,6))

    plt.scatter(
        emb_2d[:,0],
        emb_2d[:,1],
        c=labels,
        cmap="tab10",
        s=8
    )

    plt.title(f"{dataset_name} - {method_name} tSNE")

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_{method_name}_tsne.png")

    plt.close()

Sensitivity Heatmap

In [15]:
def sensitivity_analysis(G, features, labels, dataset_name):

    m_values = [4,6,8,10]

    k_values = [6,8,10,12]

    results = []

    for m in m_values:

        for k in k_values:

            print(f"Running m={m}, k={k}")

            embs = train_cplsh(
                G,
                features,
                m=m,
                k=k,
                epochs=3
            )

            scores = node_classification(embs, labels)

            avg_f1 = np.mean([x[1] for x in scores])

            results.append((m,k,avg_f1))

    df = pd.DataFrame(results, columns=["m","k","F1"])

    pivot = df.pivot(index="m", columns="k", values="F1")

    plt.figure(figsize=(8,6))

    sns.heatmap(
        pivot,
        annot=True,
        cmap="viridis"
    )

    plt.title(f"{dataset_name} Sensitivity Heatmap")

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_sensitivity_heatmap.png")

    plt.close()

Main Experiment

In [16]:
datasets = [
    "Cora",
    "Citeseer"
]

all_results = []

for dataset_name in datasets:

    print(f"\nRunning on {dataset_name}")

    dataset = Planetoid(
        root=f"./data/{dataset_name}",
        name=dataset_name
    )

    data = dataset[0]

    G = pyg_utils.to_networkx(
        data,
        to_undirected=True
    )

    features = data.x.numpy()

    labels = data.y.numpy()

    ##################################################
    # Degree Distribution
    ##################################################

    plot_degree_distribution(G, dataset_name)

    ##################################################
    # DeepWalk
    ##################################################

    print("\nTraining DeepWalk")

    dw_embs, dw_time, dw_mem = measure_time_memory(
        train_deepwalk,
        G
    )

    ##################################################
    # CP-LSH
    ##################################################

    print("\nTraining CP-LSH")

    cp_embs, cp_time, cp_mem = measure_time_memory(
        train_cplsh,
        G,
        features
    )

    ##################################################
    # Classification
    ##################################################

    dw_f1 = node_classification(dw_embs, labels)

    cp_f1 = node_classification(cp_embs, labels)

    ##################################################
    # Tail Analysis
    ##################################################

    dw_tail, dw_head = tail_head_analysis(
        G,
        dw_embs,
        labels
    )

    cp_tail, cp_head = tail_head_analysis(
        G,
        cp_embs,
        labels
    )

    ##################################################
    # Link Prediction
    ##################################################

    dw_auc = link_prediction(G, dw_embs)

    cp_auc = link_prediction(G, cp_embs)

    ##################################################
    # t-SNE
    ##################################################

    tsne_plot(dw_embs, labels, dataset_name, "DeepWalk")

    tsne_plot(cp_embs, labels, dataset_name, "CPLSH")

    ##################################################
    # Save Results
    ##################################################

    all_results.append([
        dataset_name,
        dw_time,
        dw_mem,
        dw_auc,
        cp_time,
        cp_mem,
        cp_auc
    ])

    ##################################################
    # Training Ratio Plot
    ##################################################

    plt.figure(figsize=(7,5))

    plt.plot(
        [x[0]*100 for x in dw_f1],
        [x[1] for x in dw_f1],
        marker='o',
        label="DeepWalk"
    )

    plt.plot(
        [x[0]*100 for x in cp_f1],
        [x[1] for x in cp_f1],
        marker='o',
        label="CP-LSH"
    )

    plt.xlabel("% Training Nodes")

    plt.ylabel("Micro-F1")

    plt.title(f"{dataset_name} Classification")

    plt.legend()

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_classification_curve.png")

    plt.close()

    ##################################################
    # Tail vs Head Plot
    ##################################################

    plt.figure(figsize=(6,5))

    x = np.arange(2)

    width = 0.35

    plt.bar(
        x - width/2,
        [dw_tail, dw_head],
        width,
        label="DeepWalk"
    )

    plt.bar(
        x + width/2,
        [cp_tail, cp_head],
        width,
        label="CP-LSH"
    )

    plt.xticks(x, ["Tail", "Head"])

    plt.ylabel("Micro-F1")

    plt.title(f"{dataset_name} Tail vs Head")

    plt.legend()

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_tail_head.png")

    plt.close()

    ##################################################
    # Runtime + Memory
    ##################################################

    plt.figure(figsize=(6,5))

    plt.bar(
        ["DeepWalk","CP-LSH"],
        [dw_time, cp_time]
    )

    plt.ylabel("Seconds")

    plt.title(f"{dataset_name} Runtime")

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_runtime.png")

    plt.close()

    plt.figure(figsize=(6,5))

    plt.bar(
        ["DeepWalk","CP-LSH"],
        [dw_mem, cp_mem]
    )

    plt.ylabel("MB")

    plt.title(f"{dataset_name} Memory Usage")

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_memory.png")

    plt.close()

    ##################################################
    # Sensitivity
    ##################################################

    sensitivity_analysis(
        G,
        features,
        labels,
        dataset_name
    )

results_df = pd.DataFrame(
    all_results,
    columns=[
        "Dataset",
        "DW_Time",
        "DW_Memory",
        "DW_AUC",
        "CP_Time",
        "CP_Memory",
        "CP_AUC"
    ]
)

results_df.to_csv("final_results.csv", index=False)

print("\nALL EXPERIMENTS FINISHED")


Running on Cora

Training DeepWalk

Training CP-LSH
Total training pairs: 2301800
Epoch 1 Loss: 1412.3718
Epoch 2 Loss: 1145.9601
Epoch 3 Loss: 1104.3791


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


Running m=4, k=6
Total training pairs: 2301800
Epoch 1 Loss: 4677.3088
Epoch 2 Loss: 4516.3157
Epoch 3 Loss: 4509.5807
Running m=4, k=8
Total training pairs: 2301800
Epoch 1 Loss: 2608.4800
Epoch 2 Loss: 2173.3516
Epoch 3 Loss: 2101.6591
Running m=4, k=10
Total training pairs: 2301800
Epoch 1 Loss: 1488.9356
Epoch 2 Loss: 1177.1566
Epoch 3 Loss: 1131.3838
Running m=4, k=12
Total training pairs: 2301800
Epoch 1 Loss: 1436.6442
Epoch 2 Loss: 1248.6440
Epoch 3 Loss: 1224.8348
Running m=6, k=6
Total training pairs: 2301800
Epoch 1 Loss: 4032.8167
Epoch 2 Loss: 3745.2031
Epoch 3 Loss: 3718.5785
Running m=6, k=8
Total training pairs: 2301800
Epoch 1 Loss: 2095.6354
Epoch 2 Loss: 1578.3703
Epoch 3 Loss: 1482.0773
Running m=6, k=10
Total training pairs: 2301800
Epoch 1 Loss: 1413.0383
Epoch 2 Loss: 1146.0719
Epoch 3 Loss: 1105.9093
Running m=6, k=12
Total training pairs: 2301800
Epoch 1 Loss: 1422.1338
Epoch 2 Loss: 1237.5161
Epoch 3 Loss: 1212.1830
Running m=8, k=6
Total training pairs: 23018

Processing...
Done!



Training DeepWalk

Training CP-LSH
Total training pairs: 2787150
Epoch 1 Loss: 796.2134
Epoch 2 Loss: 598.9542
Epoch 3 Loss: 588.0263


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


Running m=4, k=6
Total training pairs: 2787150
Epoch 1 Loss: 4443.8315
Epoch 2 Loss: 4195.4629
Epoch 3 Loss: 4185.7802
Running m=4, k=8
Total training pairs: 2787150
Epoch 1 Loss: 1798.9311
Epoch 2 Loss: 1299.3593
Epoch 3 Loss: 1226.5271
Running m=4, k=10
Total training pairs: 2787150
Epoch 1 Loss: 845.2232
Epoch 2 Loss: 614.2265
Epoch 3 Loss: 600.7963
Running m=4, k=12
Total training pairs: 2787150
Epoch 1 Loss: 826.4774
Epoch 2 Loss: 683.8486
Epoch 3 Loss: 674.9657
Running m=6, k=6
Total training pairs: 2787150
Epoch 1 Loss: 3500.5252
Epoch 2 Loss: 3080.3009
Epoch 3 Loss: 3044.3115
Running m=6, k=8
Total training pairs: 2787150
Epoch 1 Loss: 1240.9148
Epoch 2 Loss: 778.5660
Epoch 3 Loss: 713.3856
Running m=6, k=10
Total training pairs: 2787150
Epoch 1 Loss: 802.5492
Epoch 2 Loss: 602.8879
Epoch 3 Loss: 589.4354
Running m=6, k=12
Total training pairs: 2787150
Epoch 1 Loss: 822.9914
Epoch 2 Loss: 681.6865
Epoch 3 Loss: 675.0302
Running m=8, k=6
Total training pairs: 2787150
Epoch 1 Los